# Basics &mdash; Negating Implication, and Moving Terms Across It

**Concept 9 of the Basics decomposition:** *Negating Implication, and Moving Terms Across It*

$\neg(a\Rightarrow b)\equiv(a\wedge\neg b)$; and conjuncts can be shuffled across the arrow.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Negating-Implication/Concept-Negating-Implication.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$$\neg(a \Rightarrow b) \ \equiv\ (a \wedge \neg b)$$

To refute an implication you must exhibit the antecedent **holding** while the
consequent **fails**. There is no other way, and no $\vee$ appears.

Terms can also be **moved across the arrow**, which is the manipulation the book
actually uses:

$$(a \wedge b) \Rightarrow c \ \equiv\ a \Rightarrow (b \Rightarrow c)$$
$$a \Rightarrow (b \vee c) \ \equiv\ (a \wedge \neg b) \Rightarrow c$$

The first is **currying** (Chapter 18!). The second is how a disjunctive goal becomes
an extra hypothesis &mdash; "to prove $b$ or $c$, assume $\neg b$ and prove $c$".

## 2. Definitions

### The laws

In [ ]:
IMP = lambda a, b: (not a) or b
B3 = [(a, b, c) for a in (False, True) for b in (False, True) for c in (False, True)]

def equiv3(f, g):
    return all(f(a, b, c) == g(a, b, c) for a, b, c in B3)

## 3. Tests

**Negating an implication** leaves a conjunction.

In [ ]:
for a in (False, True):
    for b in (False, True):
        print("  !(%-6s => %-6s) = %-6s   %-6s and !%-6s = %s"
              % (a, b, not IMP(a, b), a, b, a and not b))
assert all((not IMP(a, b)) == (a and not b)
           for a in (False, True) for b in (False, True))
print("\nNo 'or' anywhere: refuting a => b needs a TRUE and b FALSE.")

**Currying:** $(a\wedge b)\Rightarrow c \equiv a\Rightarrow(b\Rightarrow c)$.

In [ ]:
f = lambda a, b, c_: IMP(a and b, c_)
g = lambda a, b, c_: IMP(a, IMP(b, c_))
assert equiv3(f, g)
print("%-7s %-7s %-7s %-16s %s" % ("a", "b", "c", "(a and b) => c", "a => (b => c)"))
for a, b, c_ in B3:
    print("%-7s %-7s %-7s %-16s %s" % (a, b, c_, f(a, b, c_), g(a, b, c_)))
print("\nThis is currying -- the same law as Chapter 18, Concept 2.")

**A disjunctive goal becomes an extra hypothesis.**

In [ ]:
f = lambda a, b, c_: IMP(a, b or c_)
g = lambda a, b, c_: IMP(a and not b, c_)
assert equiv3(f, g)
print("  a => (b or c)   ==   (a and !b) => c   ?", equiv3(f, g))
print()
print("Reading: to prove 'b or c' from a, you may ASSUME !b and prove c.")
print("That is the standard move in a proof by cases.")

Two more shuffles worth knowing.

In [ ]:
pairs = [("a => (b and c)", lambda a, b, c_: IMP(a, b and c_),
          "(a=>b) and (a=>c)", lambda a, b, c_: IMP(a, b) and IMP(a, c_)),
         ("(a or b) => c",  lambda a, b, c_: IMP(a or b, c_),
          "(a=>c) and (b=>c)", lambda a, b, c_: IMP(a, c_) and IMP(b, c_))]
for n1, f1, n2, f2 in pairs:
    print("  %-18s ==  %-20s ? %s" % (n1, n2, equiv3(f1, f2)))
    assert equiv3(f1, f2)

A worked refutation, in the shape the book uses.

In [ ]:
# claim: for all n, (n is even) => (n is divisible by 4)   -- FALSE
claim = lambda n: IMP(n % 2 == 0, n % 4 == 0)
bad = [n for n in range(20) if not claim(n)]
print("counterexamples :", bad)
n = bad[0]
print("  n = %d : antecedent (even) = %s, consequent (div by 4) = %s"
      % (n, n % 2 == 0, n % 4 == 0))
assert n % 2 == 0 and n % 4 != 0
print("\nExactly the shape !(a => b) = a and !b demands.")

## 4. Exercises


1. Negate $(a \vee b) \Rightarrow (c \wedge d)$ down to literals.
2. Is $(a \Rightarrow b) \Rightarrow c$ the same as $a \Rightarrow (b \Rightarrow c)$?
3. Use the disjunctive-goal law on a proof you have written.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Basics/Concept-Negating-Implication')